# LoRA Fine-Tuning **Nemotron-3-Ultra** on Text2SQL with NeMo AutoModel

> An end-to-end cookbook that takes you from a fresh checkout to a
> fine-tuned, deployable model — and explains *why* each knob is set the way it is.

**Model:** `Nemotron-3-Ultra-550B-A55B-BF16` — a 550B-parameter Mixture-of-Experts
model with ~55B active parameters per token and a Multi-Token-Prediction (MTP) head.

**Task:** [BIRD-SQL](https://bird-bench.github.io/) text-to-SQL — turn a natural-language
question about a database into an executable SQL query. It is a great teaching task
because the output is *verifiable*: we can run the generated SQL and measure execution
accuracy, not just look at loss curves.

**Method:** Parameter-efficient fine-tuning with **LoRA** (rank 32). We freeze the 550B
base weights and train a few tens of MB of adapter weights — the only way to adapt a
model this size on a sane budget.

---

### What you'll learn
1. How NeMo AutoModel expresses a frontier-scale training run as a single YAML recipe.
2. The Ultra-specific architecture knobs — **MTP**, **MoE expert parallelism (`ep_size`)**,
   **FSDP2**, **Transformer-Engine backends** — and why LoRA must *exclude* the Mamba `out_proj`.
3. How one notebook drives **two different supercomputers** (GB200 and H100) from a single switch.
4. A complete post-training loop: **loss visualization → before/after generation →
   SQL execution accuracy → vLLM serving**.

### ⚠️ Minimum hardware & storage
This is a frontier model. Both targets are **multi-node** runs (the 550B MoE needs
`ep_size = world_size` GPUs, which spans several nodes):

| Target | Minimum allocation | `ep_size` | Launch geometry |
|---|---|---|---|
| **GB200** (primary / P0) | **4 × GB200 nodes — 16 GPUs (184 GB each)** | `16` | `--nnodes=4 --nproc-per-node=4` |
| **H100** (secondary / P1) | **32 × H100 80 GB** (4 × 8) | `32` | `--nnodes=4 --nproc-per-node=8` |

Plus the base checkpoint from NGC: **~1.1 TB on disk (BF16)**. CUDA 12.1+, Python 3.10+, `uv`, `mamba-ssm`, `causal-conv1d`, and an HF token (gated tokenizer).

> The CPU-side steps (environment prep, dataset creation, config templating, post-hoc eval /
> visualization / serving) run in-kernel. The **training step** is launched by `LAUNCHER` (§1):
> **`slurm`** by default — the notebook writes and submits an `sbatch` job (§5.3); or **`ssh`** —
> an in-notebook SSH fan-out of `torchrun` across the nodes for interactive runs (needs passwordless
> SSH between them, checked in §5.2).

## 1. Configure your run

Everything that varies between machines and users lives in this **one cell** — there are
**no hardcoded cluster paths** anywhere else in the notebook. Set these and run top-to-bottom.

The switches that matter most:
- **`COMPUTE_TARGET`** — selects the matching YAML recipe and the multi-node launch geometry
  (`N_NODES × N_PROC_PER_NODE`).
- **`LAUNCHER`** — how training is launched (§5.3): **`"slurm"`** (default — submit an `sbatch` job)
  or **`"ssh"`** (interactive in-notebook fan-out).
- **`NODES`** — only used by the **`ssh`** launcher: the hostnames of the allocation you're already
  on (set `NEMO_NODES="h1 h2 h3 h4"`, or edit the list). Slurm allocates nodes itself, so it ignores `NODES`.

> **`ep_size` ↔ world size.** Expert-parallel size must equal `N_NODES * N_PROC_PER_NODE`.
> The shipped recipes use `ep_size: 16` (GB200) and `ep_size: 32` (H100). If you change the
> GPU count you must edit **both** `ep_size` (in the YAML) **and** the launch geometry here —
> this is the one hardware-coupled knob, and the notebook will warn you if they disagree.

In [ ]:
import os
import socket
from pathlib import Path

# ============================ USER SETTINGS ============================
# --- 1. Which machine & how to launch --------------------------------
COMPUTE_TARGET   = "GB200"        # "GB200" (primary) or "H100" (secondary)
LAUNCHER         = os.environ.get("LAUNCHER", "slurm")   # "slurm" (default — sbatch) or "ssh" (in-notebook fan-out)

# --- 2. Multi-node launch geometry (must satisfy ep_size == N_NODES*N_PROC_PER_NODE)
#     GB200 minimum: 4 x 4 = 16 GPUs (ep_size 16)
#     H100  minimum: 4 x 8 = 32 GPUs (ep_size 32)
N_NODES          = 4                                    # both targets: 4 nodes minimum
N_PROC_PER_NODE  = 4 if COMPUTE_TARGET == "GB200" else 8  # GPUs/node: GB200=4, H100=8

# --- 3. Paths --------------------------------------------------------
WORKSPACE_ROOT   = Path(os.environ.get("WORKSPACE_ROOT", Path.home() / "nemotron-ultra-workspace"))
AUTOMODEL_DIR    = Path(os.environ.get("AUTOMODEL_DIR",  WORKSPACE_ROOT / "Automodel"))
DATASET_DIR      = Path(os.environ.get("DATASET_DIR",    WORKSPACE_ROOT / "data" / "text2sql"))
OUTPUT_DIR       = Path(os.environ.get("OUTPUT_DIR",     WORKSPACE_ROOT / "runs" / "ultra_text2sql_lora"))

# --- 4. NGC base model (Section 2.1 downloads it here if missing) -----
# PLACEHOLDER (early access) — confirm/replace the slug, version, and URL below with the
# final NGC model path once it's published.
NGC_MODEL         = "0767305323357365/ultra-3-ga-bf16/ea_nvidia_nemotron_3_ultra_550b_a55b_bf16_052726"
NGC_MODEL_VERSION = os.environ.get("NGC_MODEL_VERSION", "v0.1")   # version from the NGC model page
# https://registry.ngc.nvidia.com/orgs/0767305323357365/teams/ultra-3-ga-bf16/models/ea_nvidia_nemotron_3_ultra_550b_a55b_bf16_052726
MODEL_PATH        = WORKSPACE_ROOT / f"{NGC_MODEL.split('/')[-1]}_v{NGC_MODEL_VERSION}"   # where NGC puts it

# --- 5. Auth ----------------------------------------------------------
HF_TOKEN    = os.environ.get("HF_TOKEN", "")      # gated tokenizer  (or: huggingface-cli login)
NGC_API_KEY = os.environ.get("NGC_API_KEY", "")   # NGC download     (or: ngc config set)

# --- 6. Node hostnames — only needed for LAUNCHER="ssh" (Slurm allocates nodes itself) ---
#     One hostname per node; len(NODES) must equal N_NODES. Set NEMO_NODES="h1 h2 h3 h4" or edit below.
NODES = os.environ.get("NEMO_NODES", socket.gethostname()).replace(",", " ").split()
# ======================================================================

assert COMPUTE_TARGET in ("GB200", "H100"), COMPUTE_TARGET
assert LAUNCHER in ("slurm", "ssh"), LAUNCHER
CONFIG_NAME = f"nemotron_ultra_v3_text2sql_peft_{COMPUTE_TARGET.lower()}.yaml"
EXPECTED_EP = {"GB200": 16, "H100": 32}[COMPUTE_TARGET]
WORLD_SIZE  = N_NODES * N_PROC_PER_NODE

for d in (WORKSPACE_ROOT, DATASET_DIR, OUTPUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"COMPUTE_TARGET : {COMPUTE_TARGET}")
print(f"launcher       : {LAUNCHER}")
print(f"recipe         : {CONFIG_NAME}")
print(f"world size     : {N_NODES} nodes x {N_PROC_PER_NODE} = {WORLD_SIZE} GPUs")
if WORLD_SIZE != EXPECTED_EP:
    print(f"\n  WARNING: world size ({WORLD_SIZE}) != recipe ep_size ({EXPECTED_EP}).")
    print(f"  Edit `ep_size: {EXPECTED_EP}` in {CONFIG_NAME} to match, or adjust N_NODES/N_PROC_PER_NODE.")
else:
    print(f"ep_size        : {EXPECTED_EP}  (matches world size)")
print(f"\nMODEL_PATH     : {MODEL_PATH}")
print(f"AUTOMODEL_DIR  : {AUTOMODEL_DIR}")
print(f"DATASET_DIR    : {DATASET_DIR}")
print(f"OUTPUT_DIR     : {OUTPUT_DIR}")
if LAUNCHER == "ssh":
    print(f"NODES          : {NODES}  (need {N_NODES})")
    if len(NODES) != N_NODES:
        print(f"\n  WARNING: NODES has {len(NODES)} host(s) but N_NODES={N_NODES}.")
        print(f"  Set NEMO_NODES or edit NODES so it lists exactly {N_NODES} hostnames.")
else:
    print("NODES          : (n/a for slurm — the scheduler allocates nodes)")

# This cookbook directory (where the .yaml / text2sql.py templates live):
COOKBOOK_DIR = Path.cwd()
print(f"COOKBOOK_DIR   : {COOKBOOK_DIR}")

## 2. Environment setup

Three things have to be in place before training:
1. **The base checkpoint**, downloaded from NGC (~1.1 TB).
2. **The AutoModel library**, cloned and installed in its `.venv` (you prepare this once; the
   notebook runs on that `.venv` kernel).
3. **Our two artifacts copied into the AutoModel tree** — the YAML recipe and the
   `text2sql.py` dataset target — so the config's `_target_` strings resolve.

§2.1 downloads the checkpoint automatically if it's missing; §2.2 just **verifies your prepared
AutoModel repo + `.venv`**; §2.3 copies the artifacts into place.

### 2.1 Download the base model from NGC

`Nemotron-3-Ultra-550B-A55B-BF16` (~1.1 TB) lives in the NGC private registry. The cell below
downloads it **only if it isn't already on disk**, authenticating from `NGC_API_KEY` (or a prior
`ngc config set`). It lands at `MODEL_PATH` (set in §1), and the rest of the notebook uses it.

In [ ]:
import os, shutil, subprocess

# Download the base checkpoint from NGC if it isn't already on disk (~1.1 TB).
if MODEL_PATH.exists():
    print(f"✓ already downloaded: {MODEL_PATH}")
else:
    assert shutil.which("ngc"), "Install the NGC CLI first: https://org.ngc.nvidia.com/setup/installers/cli"
    if NGC_API_KEY:
        os.environ["NGC_CLI_API_KEY"] = NGC_API_KEY          # non-interactive auth (else: ngc config set)
    print(f"Downloading {NGC_MODEL}:{NGC_MODEL_VERSION} (~1.1 TB, takes a while) ...")
    subprocess.run(
        ["ngc", "registry", "model", "download-version",
         f"{NGC_MODEL}:{NGC_MODEL_VERSION}", "--dest", str(WORKSPACE_ROOT)],
        check=True,
    )
    print(f"✓ downloaded: {MODEL_PATH}")

# sanity check
print(f"  config.json: {(MODEL_PATH / 'config.json').exists()}   "
      f"safetensors shards: {len(list(MODEL_PATH.glob('*.safetensors')))}")

### 2.2 Use the AutoModel environment

This notebook **assumes you've already cloned [NVIDIA-NeMo/Automodel](https://github.com/NVIDIA-NeMo/Automodel)
to `AUTOMODEL_DIR` and built its `.venv`**, and that **you launched Jupyter with that `.venv` as the
kernel**. The cell below just verifies that.

If you haven't set it up yet, do it once in a terminal, then restart Jupyter on the `.venv` kernel:

```bash
# branch is a PLACEHOLDER (dev) — replace with the final branch/tag
git clone --branch adil/adil-test https://github.com/NVIDIA-NeMo/Automodel.git "$AUTOMODEL_DIR"
cd "$AUTOMODEL_DIR"
uv venv && source .venv/bin/activate
uv sync --frozen
pip install mamba-ssm causal-conv1d --no-build-isolation   # Nemotron Mamba kernels
```

In [ ]:
import importlib.util

# This notebook assumes AutoModel is ALREADY cloned + installed in its .venv, and that
# you launched Jupyter with that .venv as the kernel (see the setup commands above).
VENV_DIR = AUTOMODEL_DIR / ".venv"
assert AUTOMODEL_DIR.is_dir(), f"AutoModel repo not found at {AUTOMODEL_DIR} — clone it first (see above)."
assert VENV_DIR.is_dir(),      f"AutoModel .venv not found at {VENV_DIR} — create it first (see above)."
print(f"✓ AutoModel repo : {AUTOMODEL_DIR}")
print(f"✓ .venv          : {VENV_DIR}")

if importlib.util.find_spec("nemo_automodel") is not None:
    print("✓ nemo_automodel is importable in this kernel")
else:
    print("⚠ nemo_automodel not importable — start Jupyter with the AutoModel .venv kernel "
          f"({VENV_DIR}), then re-run.")

### 2.3 Copy our artifacts into the AutoModel tree

This is the key wiring step. The recipe references the dataset by its **import path**
(`nemo_automodel.components.datasets.llm.text2sql.make_text2sql_dataset`), so `text2sql.py`
has to live *inside the installed `nemo_automodel` package*. We locate the package
automatically with `importlib`.

- YAML recipe  → `AUTOMODEL_DIR/examples/llm_finetune/nemotron/`
- `text2sql.py` → `<nemo_automodel package>/components/datasets/llm/`

In [ ]:
import shutil

# (a) copy the selected recipe (and its sibling) into the examples tree
nemotron_cfg_dir = AUTOMODEL_DIR / "examples" / "llm_finetune" / "nemotron"
if nemotron_cfg_dir.exists():
    for y in ("nemotron_ultra_v3_text2sql_peft_gb200.yaml",
              "nemotron_ultra_v3_text2sql_peft_h100.yaml"):
        src = COOKBOOK_DIR / y
        if src.exists():
            shutil.copy2(src, nemotron_cfg_dir / y)
            print(f"copied  {y}  ->  {nemotron_cfg_dir}")
else:
    print(f"(AutoModel not installed yet) would copy YAMLs -> {nemotron_cfg_dir}")

# (b) copy text2sql.py INTO the installed nemo_automodel package
def nemo_automodel_llm_dir():
    import importlib.util
    spec = importlib.util.find_spec("nemo_automodel")
    if spec is None or not spec.submodule_search_locations:
        return None
    return Path(list(spec.submodule_search_locations)[0]) / "components" / "datasets" / "llm"

llm_dir = nemo_automodel_llm_dir()
if llm_dir and llm_dir.exists():
    shutil.copy2(COOKBOOK_DIR / "text2sql.py", llm_dir / "text2sql.py")
    print(f"copied  text2sql.py  ->  {llm_dir}")
else:
    print("(nemo_automodel not importable yet) — after `uv sync`, re-run this cell so")
    print("text2sql.py lands at <nemo_automodel>/components/datasets/llm/text2sql.py")

## 3. Create the Text2SQL dataset (BIRD-SQL → JSONL)

NeMo AutoModel's `make_text2sql_dataset` reads JSONL with two columns — **`input`** (the prompt)
and **`output`** (the gold target). This section turns the BIRD-SQL benchmark into
`training.jsonl` / `validation.jsonl` / `test.jsonl` under `DATASET_DIR`.

We wrap each example in a consistent instruction template (question + optional external-knowledge
"evidence" + optional schema). `answer_only_loss_mask: true` in the recipe means the model is
**only scored on the target it generates**, not on the prompt tokens.

**Reasoning:** Ultra is a reasoning model, so with `INCLUDE_REASONING=True` we also fold in
chain-of-thought traces from a reasoning-augmented BIRD mirror — the target becomes
*reasoning → `</think>` → SQL*, teaching the model to think before it answers. Plain BIRD rows
(no trace) keep an SQL-only target, so the corpus trains both behaviours.

In [ ]:
# ---- dataset-prep settings ----
BIRD_HF_DATASET      = os.environ.get("BIRD_HF_DATASET", "xu3kev/BIRD-SQL-data-train")
REASONING_HF_DATASET = os.environ.get("REASONING_HF_DATASET", "meowterspace45/bird-sql-train-with-reasoning")
# ^ Verified mirrors: columns (db_id / question / evidence / SQL [+ reasoning_trace]) match ROW_FIELDS
#   and load with a plain `load_dataset(..., split="train")`. Override either via env if you use a
#   different mirror, adjusting ROW_FIELDS to match.
VALIDATION_FRACTION = 0.05
TEST_FRACTION       = 0.02
SPLIT_SEED          = 1234
INCLUDE_EVIDENCE    = True     # include BIRD "evidence" (external knowledge) in the prompt
INCLUDE_REASONING   = True     # Ultra is a reasoning model — also fold chain-of-thought traces into the
                               # target (output = <reasoning> </think> SQL). Set False for SQL-only targets.
LIMIT               = None     # e.g. 200 for a fast smoke test; None = full dataset

ROW_FIELDS = {           # map BIRD column names -> our roles (edit if your mirror differs)
    "question":  ["question"],
    "sql":       ["SQL", "query", "sql"],
    "evidence":  ["evidence"],
    "db_id":     ["db_id", "database_id"],
    "reasoning": ["reasoning_trace"],
}

PROMPT_TEMPLATE = (
    "You are an expert data analyst. Write a single SQLite SQL query that answers the "
    "question.\n\n"
    "### Database: {db_id}\n"
    "{evidence}"
    "### Question:\n{question}\n\n"
    "### SQL:\n"
)

def _pick(row, keys):
    for k in keys:
        if k in row and row[k] not in (None, ""):
            return row[k]
    return None

def row_to_example(row):
    q   = _pick(row, ROW_FIELDS["question"])
    sql = _pick(row, ROW_FIELDS["sql"])
    if not q or not sql:
        return None
    ev   = _pick(row, ROW_FIELDS["evidence"]) if INCLUDE_EVIDENCE else None
    dbid = _pick(row, ROW_FIELDS["db_id"]) or "unknown"
    rsn  = _pick(row, ROW_FIELDS["reasoning"]) if INCLUDE_REASONING else None
    ev_block = f"### Knowledge:\n{ev}\n\n" if ev else ""
    prompt = PROMPT_TEMPLATE.format(db_id=dbid, evidence=ev_block, question=q.strip())
    # Reasoning target: emit the trace, close the think block, then the SQL. Rows without a
    # reasoning_trace (e.g. plain BIRD) just get SQL — so a mixed corpus trains both behaviours.
    output = f"{rsn.strip()}\n</think>\n\n{sql.strip()}" if rsn else sql.strip()
    return {"input": prompt, "output": output, "db_id": dbid}

print("datasets:", BIRD_HF_DATASET, "+ reasoning" if INCLUDE_REASONING else "(no reasoning)",
      "| evidence:", INCLUDE_EVIDENCE, "| limit:", LIMIT)

In [ ]:
import json, random
from datasets import load_dataset

rows = list(load_dataset(BIRD_HF_DATASET, split="train"))
print(f"loaded {len(rows)} examples from {BIRD_HF_DATASET}")
if INCLUDE_REASONING:
    try:
        r = list(load_dataset(REASONING_HF_DATASET, split="train"))
        rows += r
        print(f"+ {len(r)} reasoning examples from {REASONING_HF_DATASET}")
    except Exception as e:
        print(f"(skipping reasoning source: {e})")

random.Random(SPLIT_SEED).shuffle(rows)        # shuffle before LIMIT so a smoke test samples both sources
if LIMIT:
    rows = rows[:LIMIT]

examples = [e for e in (row_to_example(r) for r in rows) if e]

n = len(examples)
n_val  = int(n * VALIDATION_FRACTION)
n_test = int(n * TEST_FRACTION)
val   = examples[:n_val]
test  = examples[n_val:n_val + n_test]
train = examples[n_val + n_test:]

def write_jsonl(path, rows, keep_dbid=False):
    with open(path, "w") as f:
        for r in rows:
            rec = {"input": r["input"], "output": r["output"]}
            if keep_dbid:
                rec["db_id"] = r["db_id"]
            f.write(json.dumps(rec) + "\n")

TRAIN_JSONL = DATASET_DIR / "training.jsonl"
VAL_JSONL   = DATASET_DIR / "validation.jsonl"
TEST_JSONL  = DATASET_DIR / "test.jsonl"
write_jsonl(TRAIN_JSONL, train)
write_jsonl(VAL_JSONL,   val)
write_jsonl(TEST_JSONL,  test, keep_dbid=True)   # keep db_id for execution-accuracy eval

n_reason = sum("</think>" in e["output"] for e in examples)
print(f"train: {len(train)}  val: {len(val)}  test: {len(test)}  (total {n}; {n_reason} with reasoning)")
print("wrote:", TRAIN_JSONL, VAL_JSONL, TEST_JSONL, sep="\n  ")
print("\n--- sample prompt+target ---\n" + train[0]["input"] + train[0]["output"][:400])

## 4. Understanding the Ultra recipe

Before launching, it's worth understanding *why* the YAML looks the way it does. The whole run
is described declaratively — `finetune.py` just instantiates the `_target_` objects. Here are
the Ultra-specific pieces.

### Multi-Token Prediction (MTP)
```yaml
num_nextn_predict_layers: 2
mtp_use_repeated_layer: true
mtp_loss_scaling_factor: 0.1
```
The MTP head predicts the **next 2 tokens** in parallel (great for later speculative decoding).
`mtp_use_repeated_layer: true` reuses **one physical MTP layer** across both prediction depths
(weight-tied, mirrors Megatron's `--mtp-use-repeated-layer`), so the parameter count matches a
1-depth MTP. The block-type pattern is auto-resolved from Ultra's `mtp_layers_block_type`.

### Mixture-of-Experts + expert parallelism
```yaml
backend: { experts: gmm, dispatcher: hybridep|deepep }
distributed: { strategy: fsdp2, ep_size: 16|32 }
```
Ultra is a sparse MoE (550B total / ~55B active). Experts are sharded across GPUs with
**expert parallelism**; `ep_size` **must equal the world size**. The token **dispatcher** differs
by fabric — `hybridep` on GB200's NVLink-rich domain, `deepep` on H100. `experts: gmm` uses grouped
GEMM for the expert FFNs. Everything runs under **FSDP2**.

### Why LoRA excludes `out_proj`
```yaml
peft: { exclude_modules: ["*.out_proj"], dim: 32, alpha: 32, use_triton: true }
```
Nemotron is a **hybrid Mamba-Transformer**. The Mamba layers use custom kernels that consume
`out_proj.weight` directly, so a LoRA wrapper there would break them — we exclude those modules
and adapt everything else.

### One recipe, two supercomputers

| Field | GB200 (P0) | H100 (P1) | why |
|---|---|---|---|
| `backend.dispatcher` | `hybridep` | `deepep` | fabric-specific MoE token routing |
| `backend.enable_fsdp_optimizations` | `true` | `false` | GB200 NVLink domain enables extra fusions |
| `distributed.ep_size` | `16` | `32` | = world size (16 vs 32 GPUs) |
| `moe.reshard_after_forward` | `false` | `true` | memory/throughput trade-off per node size |
| launch | `--nnodes=4 --nproc-per-node=4` | `--nnodes=4 --nproc-per-node=8` | topology |

> **Packing note (validate on first run).** These recipes keep the validated `packed_sequence`
> (THD) path with `packed_sequence_thd_collater`, because Ultra's Transformer-Engine attention is
> validated on the THD `(1, T)` code path. We only swapped the dataset `_target_` to Text2SQL. If
> you hit a collation error, the fallback is to drop the `packed_sequence` block and set
> `seq_length: 4096` on the dataset (the non-packed Text2SQL path used by the Super cookbook).

> **OOM & first-run cost.** `packed_sequence_size` is also your main **memory** lever — if you hit a
> GPU OOM, lower it (e.g. `1024`) until it fits, then raise it back; it changes only per-step
> activation memory, not correctness. Separately, the **first** run is slow to *start* because the
> ~1 TB checkpoint loads cold from disk and the TE/compile kernels autotune; later runs reuse the OS
> page cache + the compile/autotune disk cache, so startup is much faster. That caching speeds
> **startup**, it does **not** free GPU memory — so it isn't what resolves an OOM (the packed-size
> knob is). With `INCLUDE_REASONING=True` the targets are long, so prefer a *larger* packed size
> (e.g. `4096`) over a smaller one to avoid truncating reasoning traces.

## 5. Fine-tune

Four steps: 
 - (1) bake the paths from Section 1 into the recipe, 
 - (2) check launch readiness for your chosen `LAUNCHER`, 
 - (3) launch the multi-node run — **Slurm `sbatch` by default** (or the in-notebook SSH fan-out), 
 - (4) plot the loss curve from the log.

### 5.1 Substitute paths into the recipe

In [ ]:
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Always start from the PRISTINE cookbook template (this notebook never writes to COOKBOOK_DIR),
# so re-running this cell after changing a path re-substitutes from fresh tokens. Fall back to the
# repo copy only if the cookbook copy isn't found.
template_path = COOKBOOK_DIR / CONFIG_NAME
if not template_path.exists():
    template_path = AUTOMODEL_DIR / "examples" / "llm_finetune" / "nemotron" / CONFIG_NAME

text = template_path.read_text()
subs = {
    "__PRETRAINED_MODEL_PATH__": str(MODEL_PATH),
    "__TRAIN_JSONL__":           str(TRAIN_JSONL),
    "__VAL_JSONL__":             str(VAL_JSONL),
    "__CHECKPOINT_DIR__":        str(CHECKPOINT_DIR),
}
for k, v in subs.items():
    text = text.replace(k, v)

RESOLVED_CONFIG = OUTPUT_DIR / CONFIG_NAME
RESOLVED_CONFIG.write_text(text)

remaining = [k for k in subs if k in text]
print("template            :", template_path)
print("wrote resolved config:", RESOLVED_CONFIG)
print("unsubstituted tokens :", remaining or "none ✅")
# sync the resolved config into the repo so §5.3's relative `-c` path resolves to it
repo_cfg = AUTOMODEL_DIR / "examples" / "llm_finetune" / "nemotron" / CONFIG_NAME
if repo_cfg.parent.exists():
    repo_cfg.write_text(text)
    print("synced to repo      :", repo_cfg)

### 5.2 Check launch readiness

A quick pre-flight for whichever `LAUNCHER` you picked in §1:
- **`slurm`** (default) — confirms `sbatch`/`srun`/`scontrol` are on `PATH` and shows your allocation
  if you're already inside one (otherwise §5.3 submits a fresh job).
- **`ssh`** — confirms each node in `NODES` is reachable over passwordless SSH and reports its GPUs.

In [ ]:
import subprocess, shutil

if LAUNCHER == "slurm":
    tools = {t: shutil.which(t) for t in ("sbatch", "srun", "scontrol")}
    for t, p in tools.items():
        print(f"{t:>9}: {'ok' if p else 'MISSING'}")
    nodelist = os.environ.get("SLURM_JOB_NODELIST")
    if nodelist:
        print(f"\ninside an allocation: {nodelist}")
    else:
        print("\nno active allocation — §5.3 submits a fresh job via sbatch (--nodes is set for you).")
    if not tools["sbatch"]:
        print("\n  sbatch not found — is Slurm available here? (or set LAUNCHER='ssh' in §1)")
else:  # ssh
    all_ok = True
    for node in NODES:
        r = subprocess.run(
            ["ssh", "-o", "BatchMode=yes", "-o", "ConnectTimeout=5", node,
             "bash", "-lc", "nvidia-smi --query-gpu=name --format=csv,noheader | wc -l"],
            capture_output=True, text=True)
        if r.returncode == 0:
            ngpu = r.stdout.strip().splitlines()[-1] if r.stdout.strip() else "?"
            print(f"{node:>24}: OK    gpus={ngpu}")
        else:
            all_ok = False
            print(f"{node:>24}: FAIL  ({r.stderr.strip()[:80]})")
    if not all_ok:
        print("\n  Some nodes are unreachable over SSH. Fix passwordless SSH between the allocated")
        print("  nodes (or correct NODES in §1) before launching in §5.3.")
    elif len(NODES) != N_NODES:
        print(f"\n  NODES lists {len(NODES)} host(s) but N_NODES={N_NODES} — they must match.")

print(f"\nlauncher={LAUNCHER}; world size {WORLD_SIZE} ({N_NODES} x {N_PROC_PER_NODE}) "
      f"for {COMPUTE_TARGET}; ep_size={EXPECTED_EP}.")

### 5.3 Launch training

Launch is selected by **`LAUNCHER`** (set in §1):

- **`"slurm"` (default).** Writes an `sbatch` script (`srun` fans out, `torchrun` per node, `scontrol`
  resolves the rendezvous host, `$SLURM_NODEID` the rank) and — if `RUN_TRAINING=True` — submits it
  with `sbatch`. The job runs **detached** under the scheduler: robust for a long 550B run (queueing,
  walltime, process cleanup, survives a dropped connection). Follow it with `tail -f` on the per-job
  log. The scheduler allocates the nodes, so you don't need `NODES`.
- **`"ssh"` (alternative — interactive).** Launches the run **live from this notebook**: one `torchrun`
  is SSHed to each node in `NODES`, sharing a c10d rendezvous on `NODES[0]`; output streams into the
  cell and is teed to `OUTPUT_DIR/train.log`. Good when you're already on an interactive allocation
  and want to watch it (needs passwordless SSH between nodes — checked in 5.2). Note: it's tied to the
  kernel's lifetime — if the kernel dies, the run dies.

Both reach the same world (`N_NODES × N_PROC_PER_NODE`). Flip `RUN_TRAINING=True` to submit (slurm) /
launch (ssh).

In [ ]:
import subprocess, threading, shlex

RUN_TRAINING = False     # set True to actually launch (ssh) / submit (slurm). LAUNCHER is chosen in §1.

RDZV_PORT = 29500
RDZV_ID   = "ultra_text2sql"
ENTRY     = "examples/llm_finetune/finetune.py"
cfg_rel   = f"examples/llm_finetune/nemotron/{CONFIG_NAME}"
LOG_PATH  = OUTPUT_DIR / "train.log"


def _launch_ssh():
    """One torchrun SSHed per node, shared c10d rendezvous on NODES[0]; streams to the cell + log."""
    assert len(NODES) == N_NODES, f"NODES has {len(NODES)} host(s) but N_NODES={N_NODES}"
    master = NODES[0]

    def remote_cmd(rank):
        inner = (
            f"cd {shlex.quote(str(AUTOMODEL_DIR))} && source .venv/bin/activate && "
            f"export DATASET_DIR={shlex.quote(str(DATASET_DIR))} && "
            f"torchrun --nnodes={N_NODES} --nproc-per-node={N_PROC_PER_NODE} "
            f"--rdzv-backend=c10d --rdzv-endpoint={master}:{RDZV_PORT} --rdzv-id={RDZV_ID} "
            f"--node-rank={rank} {ENTRY} -c {cfg_rel}"
        )
        return ["ssh", "-o", "StrictHostKeyChecking=no", NODES[rank], "bash", "-lc", inner]

    def stream(proc, tag, logf, lock):
        for line in proc.stdout:
            with lock:
                print(f"[{tag}] {line}", end="")
                logf.write(f"[{tag}] {line}"); logf.flush()

    print(f"SSH launch | {N_NODES} nodes | rendezvous {master}:{RDZV_PORT} | log {LOG_PATH}\n")
    procs, threads, lock = [], [], threading.Lock()
    with open(LOG_PATH, "w") as logf:
        for rank, node in enumerate(NODES):
            p = subprocess.Popen(remote_cmd(rank), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                 text=True, bufsize=1, encoding="utf-8", errors="replace")
            procs.append(p)
            t = threading.Thread(target=stream, args=(p, node, logf, lock), daemon=True)
            t.start(); threads.append(t)
        try:
            for t in threads: t.join()
            rcs = {n: p.wait() for n, p in zip(NODES, procs)}
            print("\nexit codes:", rcs)
            print("✅ all nodes exited 0" if set(rcs.values()) == {0}
                  else "⚠️  non-zero exit on some node — check the log above")
        except KeyboardInterrupt:
            print("\nInterrupted — terminating torchrun on all nodes...")
            for p in procs: p.terminate()
            for node in NODES:
                subprocess.run(["ssh", "-o", "StrictHostKeyChecking=no", node, "pkill", "-f", "torchrun"])
            print("torn down.")


def _launch_slurm():
    """Write an sbatch script (srun fans out, torchrun per node) and submit it if RUN_TRAINING."""
    sbatch = f"""#!/bin/bash
#SBATCH --job-name=ultra_text2sql_lora
#SBATCH --nodes={N_NODES}
#SBATCH --ntasks-per-node=1
#SBATCH --gpus-per-node={N_PROC_PER_NODE}
#SBATCH --time=04:00:00
#SBATCH --output={OUTPUT_DIR / 'train_%j.log'}

cd "{AUTOMODEL_DIR}"
source .venv/bin/activate
export DATASET_DIR="{DATASET_DIR}"
MASTER_ADDR=$(scontrol show hostnames "$SLURM_JOB_NODELIST" | head -n 1)
srun --ntasks-per-node=1 bash -c '
  torchrun --nnodes={N_NODES} --nproc-per-node={N_PROC_PER_NODE} \\
    --rdzv-backend=c10d --rdzv-endpoint="'"$MASTER_ADDR"'":{RDZV_PORT} --rdzv-id={RDZV_ID} \\
    --node-rank=$SLURM_NODEID \\
    {ENTRY} -c {cfg_rel}'
"""
    sbatch_path = OUTPUT_DIR / "submit_ultra_text2sql.sbatch"
    sbatch_path.write_text(sbatch)
    print("wrote sbatch script:", sbatch_path)
    if RUN_TRAINING:
        r = subprocess.run(["sbatch", str(sbatch_path)], capture_output=True, text=True)
        print((r.stdout or r.stderr).strip())
        print(f"follow logs: tail -f {OUTPUT_DIR}/train_<jobid>.log")
    else:
        print(f"RUN_TRAINING=False — submit it yourself with:  sbatch {sbatch_path}")


if LAUNCHER == "slurm":
    _launch_slurm()
elif LAUNCHER == "ssh":
    if RUN_TRAINING:
        _launch_ssh()
    else:
        print("RUN_TRAINING=False — dry run (LAUNCHER='ssh').")
        print(f"  nodes     : {NODES}")
        print(f"  rendezvous: {NODES[0]}:{RDZV_PORT}  (id={RDZV_ID})")
        print(f"  per-node  : torchrun --nnodes={N_NODES} --nproc-per-node={N_PROC_PER_NODE} "
              f"--node-rank=<i> {ENTRY} -c {cfg_rel}")
        print(f"  log       : {LOG_PATH}")
else:
    raise ValueError(f"LAUNCHER must be 'slurm' or 'ssh', got {LAUNCHER!r}")

### 5.4 Visualize training convergence

Run this **after** the job finishes (or while it streams to `train.log`). We scrape the loss from
the log and plot it — a quick read on whether LoRA is learning.

In [ ]:
import re
import matplotlib.pyplot as plt

LOG_PATH = OUTPUT_DIR / "train.log"
steps, losses, val_steps, val_losses = [], [], [], []
loss_re = re.compile(r"step[^0-9]*(\d+).*?loss[:=]\s*([0-9.]+)", re.I)
val_re  = re.compile(r"val[^0-9]*?(?:step[^0-9]*(\d+)).*?loss[:=]\s*([0-9.]+)", re.I)

if LOG_PATH.exists():
    for line in LOG_PATH.read_text(errors="ignore").splitlines():
        m = val_re.search(line)
        if m:
            val_steps.append(int(m.group(1))); val_losses.append(float(m.group(2))); continue
        m = loss_re.search(line)
        if m:
            steps.append(int(m.group(1))); losses.append(float(m.group(2)))

    fig, ax = plt.subplots(figsize=(8, 4))
    if steps:     ax.plot(steps, losses, label="train loss", lw=1.5)
    if val_steps: ax.plot(val_steps, val_losses, "o-", label="val loss")
    ax.set_xlabel("step"); ax.set_ylabel("loss"); ax.set_title("Nemotron-3-Ultra LoRA / Text2SQL")
    ax.legend(); ax.grid(alpha=.3); plt.tight_layout(); plt.show()
    print(f"parsed {len(steps)} train points, {len(val_steps)} val points")
else:
    print(f"No log yet at {LOG_PATH}. Run the training job (5.3) first.")
    print("Tip: the regexes above are intentionally loose — adjust them to your log format.")

## 6. Evaluate: generate SQL & measure execution accuracy

The real test of a Text2SQL model is whether its SQL **runs and returns the right rows**. We:
1. resolve the latest LoRA adapter checkpoint,
2. load `base + adapter` and generate SQL for held-out questions (before vs after),
3. (optionally) execute predicted vs gold SQL against the BIRD SQLite databases and score
   **execution accuracy**.

### 6.1 Resolve the latest adapter checkpoint

In [ ]:
def resolve_latest_adapter(ckpt_root: Path):
    latest = ckpt_root / "LATEST"
    if latest.exists():
        target = latest.resolve()
        cand = target / "model"
        return cand if cand.exists() else target
    steps = sorted(ckpt_root.glob("*step_*"),
                   key=lambda p: int(re.findall(r"step_(\d+)", p.name)[-1]) if re.findall(r"step_(\d+)", p.name) else -1)
    if steps:
        cand = steps[-1] / "model"
        return cand if cand.exists() else steps[-1]
    return None

ADAPTER_DIR = resolve_latest_adapter(CHECKPOINT_DIR)
print("adapter dir:", ADAPTER_DIR or f"(none found under {CHECKPOINT_DIR} — train first)")

### 6.2 Before vs after — generate on held-out questions

> Loading a 550B model for inference is itself a multi-GPU job; run this on a GPU node.
> For a lighter check you can point `BASE_FOR_INFER` at a smaller Nemotron while you wire the flow.

In [ ]:
RUN_INFERENCE = False   # set True on a GPU node with the model loaded

def load_test(n=5):
    rows = [json.loads(l) for l in open(TEST_JSONL)]
    return rows[:n]

if RUN_INFERENCE and ADAPTER_DIR:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel

    BASE_FOR_INFER = str(MODEL_PATH)
    tok = AutoTokenizer.from_pretrained(BASE_FOR_INFER, trust_remote_code=True)
    base = AutoModelForCausalLM.from_pretrained(
        BASE_FOR_INFER, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
    model = PeftModel.from_pretrained(base, str(ADAPTER_DIR))
    model.eval()

    def gen(prompt, m):
        ids = tok(prompt, return_tensors="pt").to(m.device)
        out = m.generate(**ids, max_new_tokens=256, do_sample=False)
        return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)

    for ex in load_test(3):
        print("Q:", ex["input"].splitlines()[-2][:120])
        print("  gold :", ex["output"])
        print("  base :", gen(ex["input"], base).strip()[:200])
        print("  lora :", gen(ex["input"], model).strip()[:200])
        print("-" * 80)
else:
    print("Set RUN_INFERENCE=True on a GPU node (and train first) to see before/after generations.")

### 6.3 Execution accuracy + visualization

Execution accuracy = fraction of predicted queries whose result set matches the gold query's,
run against the actual BIRD SQLite DBs. Point `BIRD_DB_ROOT` at the BIRD `dev_databases`
(each `db_id` has a `<db_id>/<db_id>.sqlite`). We plot a base-vs-LoRA accuracy bar.

In [ ]:
import sqlite3
BIRD_DB_ROOT = Path(os.environ.get("BIRD_DB_ROOT", WORKSPACE_ROOT / "bird" / "dev_databases"))

def exec_query(db_id, sql):
    db = BIRD_DB_ROOT / db_id / f"{db_id}.sqlite"
    if not db.exists():
        return None
    try:
        con = sqlite3.connect(str(db)); cur = con.cursor()
        cur.execute(sql); rows = cur.fetchall(); con.close()
        return set(map(tuple, rows))
    except Exception:
        return None

def execution_match(db_id, pred_sql, gold_sql):
    p, g = exec_query(db_id, pred_sql), exec_query(db_id, gold_sql)
    return (p is not None) and (g is not None) and (p == g)

# `predictions` would be filled by 6.2 (list of dicts: db_id, gold, base_sql, lora_sql).
# Here we just show the scoring + plotting shape so it runs even before a GPU pass:
predictions = globals().get("predictions", [])
if predictions and BIRD_DB_ROOT.exists():
    import matplotlib.pyplot as plt
    base_acc = sum(execution_match(p["db_id"], p["base_sql"], p["gold"]) for p in predictions) / len(predictions)
    lora_acc = sum(execution_match(p["db_id"], p["lora_sql"], p["gold"]) for p in predictions) / len(predictions)
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.bar(["base", "LoRA"], [base_acc, lora_acc], color=["#999", "#76b900"])
    ax.set_ylim(0, 1); ax.set_ylabel("execution accuracy")
    ax.set_title("BIRD-SQL execution accuracy");
    for i, v in enumerate([base_acc, lora_acc]): ax.text(i, v + .02, f"{v:.0%}", ha="center")
    plt.tight_layout(); plt.show()
else:
    print("Fill `predictions` from Section 6.2 and set BIRD_DB_ROOT to the BIRD dev_databases dir.")
    print("Then this cell scores execution accuracy (base vs LoRA) and plots the comparison.")

## 7. Deploy with vLLM

For production serving, vLLM can load the **base model + LoRA adapter** without merging (swap
adapters per request), or you can **merge** the adapter into a standalone checkpoint for
adapter-free serving.

### 7.1 Serve base + LoRA adapter

In [ ]:
vllm_serve = f"""# serve the base model with LoRA enabled (adapter applied per-request)
vllm serve "{MODEL_PATH}" \\
  --enable-lora \\
  --max-lora-rank 32 \\
  --lora-modules ultra-sql={ADAPTER_DIR} \\
  --trust-remote-code \\
  --tensor-parallel-size {N_PROC_PER_NODE}

# then query it (OpenAI-compatible API), selecting the adapter via "model":
#   curl http://localhost:8000/v1/completions -H 'Content-Type: application/json' -d '{{
#     "model": "ultra-sql", "prompt": "<your Text2SQL prompt>", "max_tokens": 256 }}'
"""
print(vllm_serve)

### 7.2 (Optional) Merge the adapter into a standalone checkpoint

In [ ]:
RUN_MERGE = False   # set True on a GPU node to materialize a merged checkpoint

MERGED_DIR = OUTPUT_DIR / "merged"
if RUN_MERGE and ADAPTER_DIR:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    tok = AutoTokenizer.from_pretrained(str(MODEL_PATH), trust_remote_code=True)
    base = AutoModelForCausalLM.from_pretrained(
        str(MODEL_PATH), torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
    merged = PeftModel.from_pretrained(base, str(ADAPTER_DIR)).merge_and_unload()
    merged.save_pretrained(str(MERGED_DIR), safe_serialization=True)
    tok.save_pretrained(str(MERGED_DIR))
    print("merged checkpoint ->", MERGED_DIR)
else:
    print(f"Set RUN_MERGE=True (GPU node) to merge -> {MERGED_DIR}")
    print("Adapter-free serving then becomes:  vllm serve", MERGED_DIR, "--trust-remote-code")

## 8. Summary & next steps

You ran the full lifecycle for a frontier MoE model:

1. **Configured** one notebook to target either **GB200 (P0)** or **H100 (P1)** from a single switch.
2. **Downloaded** `Nemotron-3-Ultra-550B-A55B-BF16` from NGC and installed AutoModel, copying the
   recipe and `text2sql.py` into the right places.
3. **Built** a BIRD-SQL Text2SQL dataset (`input`/`output` JSONL), with reasoning traces folded in.
4. **Understood** the Ultra recipe — MTP, MoE expert parallelism, FSDP2, TE backends, and the
   Mamba `out_proj` LoRA exclusion.
5. **Fine-tuned** with LoRA, launching the multi-node run via **Slurm `sbatch`** (default) — or the
   in-notebook SSH fan-out — and **visualized** loss.
6. **Evaluated** with before/after generations and **SQL execution accuracy**.
7. **Deployed** with vLLM (adapter-swap or merged).

### Where to go next
- **Scale the data / steps** — set `max_steps` higher (the recipe runs a full epoch by default).
- **Sweep LoRA** — try different `dim`/`alpha` (e.g. 16/32, 64/16) and compare execution accuracy.
- **Tune reasoning** — adjust the plain-vs-reasoning mix (`INCLUDE_REASONING`) and see its effect on
  both execution accuracy and output length.
- **Speculative decoding** — Ultra's MTP head pairs naturally with a draft-verify decode loop.
- **Quantize for serving** — explore FP8/INT4 export paths for the merged checkpoint.

> Found an issue with the THD packing + Text2SQL combination? See the *packing note* in Section 4
> for the non-packed fallback.